In [1]:
import numpy as np

In [2]:
GLOBAL_SEED = 24
rng = np.random.default_rng(GLOBAL_SEED)

def reset_seed(seed = GLOBAL_SEED):
    global rng
    rng = np.random.default_rng(seed)

In [3]:
class WeightInitializer:       
    @staticmethod
    def zeros(shape):
        """Initialize weights with all zeros."""
        return np.zeros(shape)

    @staticmethod
    def uniform(shape, low=-0.05, high=0.05):
        """Initialize weights with a uniform distribution."""
        return rng.uniform(low, high, shape)

    @staticmethod
    def normal(shape, mean=0.0, std=0.05):
        """Initialize weights with a normal distribution."""
        return rng.normal(mean, std, shape)

    @staticmethod
    def xavier(shape, fan_in, fan_out):
        """Initialize weights using Xavier (Glorot) initialization."""
        limit = np.sqrt(6 / (fan_in + fan_out))
        return rng.uniform(-limit, limit, shape)

    @staticmethod
    def kaiming(shape, fan_in):
        """Initialize weights using Kaiming (He) initialization."""
        limit = np.sqrt(6 / fan_in)
        return np.random.normal(-limit, limit, size=shape)
    
    @staticmethod
    def initialize_weights(shape, weights_type, n_in, n_out):
        weights_type = weights_type.strip().lower()
        
        if weights_type == 'uniform':
            return WeightInitializer.uniform(shape)
        elif weights_type == 'normal':
            return WeightInitializer.normal(shape)   
        elif weights_type == 'xavier':
            return WeightInitializer.xavier(shape, n_in, n_out)  
        elif weights_type == 'kaiming':
            return WeightInitializer.normal(shape, n_in)
        else:
            weights_type = 'zero'
            return WeightInitializer.zeros(shape)

In [155]:
# -------------------------
# Activation functions
# -------------------------

class ReLU:
    @staticmethod
    def activate(x):
        return np.maximum(0, x)
    
    @staticmethod
    def derivative(x):
        return (x > 0).astype(float)

class Sigmoid:
    @staticmethod
    def activate(x):
        return 1 / (1 + np.exp(-x))

    @staticmethod
    def derivative(x):
        s = Sigmoid.activate(x)
        return s * (1 - s)
    
class Linear:
    @staticmethod
    def activate(x):
        return x

    @staticmethod
    def derivative(x):
        return np.ones_like(x)

class Tanh:
    @staticmethod
    def activate(x):
        return np.tanh(x)
    
    @staticmethod
    def derivative(x):
        return 1 - np.tanh(x)**2

class Softmax:
    @staticmethod
    def activate(x):
        # e = np.exp(x - np.max(x))
        # return e / np.sum(e)
        exp_x = np.exp(x - np.max(x))  # stability trick
        return exp_x / np.sum(exp_x)



# def softmax(z):
#     """
#     Compute softmax probabilities for each row of input z.
#     z: shape (batch_size, num_classes)
#     """
#     z_shifted = z - np.max(z, axis=1, keepdims=True)  # for numerical stability
#     exp_z = np.exp(z_shifted)
#     return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# def cross_entropy_loss(y_true, y_pred, epsilon=1e-12):
#     """
#     Compute cross-entropy loss.
#     y_true: one-hot encoded true labels, shape (batch_size, num_classes)
#     y_pred: predicted probabilities from softmax, shape (batch_size, num_classes)
#     """
#     y_pred = np.clip(y_pred, epsilon, 1. - epsilon)  # avoid log(0)
#     loss = -np.sum(y_true * np.log(y_pred), axis=1)
#     return np.mean(loss)

In [156]:
# -------------------------
# Loss functions
# -------------------------

class MSELoss:  # Do not use with Softmax activation!!
    @staticmethod
    def loss(y_pred, y_true):
        return np.mean((y_pred - y_true)**2)

    @staticmethod
    def output_delta(y_pred, y_true, z, activation):
        return (y_pred - y_true) * activation.derivative(z)  


class CrossEntropyLoss:  # Only use with Softmax activation!!
    @staticmethod
    def loss(y_pred, y_true):
        # p = Softmax.activate(y_pred)
        # return -np.sum(y_true * np.log(p + 1e-9))
        epsilon = 1e-12
        y_pred = np.clip(y_pred, epsilon, 1. - epsilon)
        
        # Compute cross-entropy
        loss = -np.sum(y_true * np.log(y_pred), axis=1)
        return np.mean(loss)


    @staticmethod
    def output_delta(y_pred, y_true, z, activation):  # activation should be Softmax
        if activation != Softmax:
            raise ValueError("Cross Entropy Loss should be used with Softmax activation in the output layer.")
        return activation.activate(y_pred) - y_true  # Softmax + Cross Entropy has a simplified formula for delta

In [157]:
# -------------------------
# Linear Layer
# -------------------------

class LinearLayer:
    def __init__(self, n_in, n_out, weights_type):
        self.shape = (n_in, n_out)
        
        # self.W = np.random.randn(n_in, n_out) * np.sqrt(2/n_in)
        self.W = WeightInitializer.initialize_weights(self.shape, weights_type, n_in, n_out)
        reset_seed()
        self.b = np.zeros((n_out,))

    def forward(self, x):
        self.x = x          # store input
        self.z = x @ self.W + self.b  # store pre-activation
        return self.z

In [158]:
# -------------------------
# MLP with Delta-Based Backprop
# -------------------------


class MLP:
    def __init__(
        self, layer_sizes, activations, loss="mse", lr=0.01, weights_type="xavier"
    ):
        self.layers = []
        self.activations = []

        self.weights_type = weights_type
        for i in range(len(layer_sizes) - 1):
            self.layers.append(
                LinearLayer(layer_sizes[i], layer_sizes[i + 1], weights_type)
            )
            if activations[i] is not None:
                self.activations.append(activations[i])
            else:
                self.activations.append(None)

        self.lr = lr
        self.loss_fn = CrossEntropyLoss if loss == "cross_entropy" else MSELoss

    def forward(self, x):
        out = x
        self.z = []  # pre-activation outputs
        self.a = [x]  # post-activation outputs, starting with input

        for layer, activation in zip(self.layers, self.activations):
            z = layer.forward(out)
            self.z.append(z)

            out = z if activation is None else activation.activate(z)
            self.a.append(out)

        return out

    def backward(self, y_true):
        deltas = [None] * len(self.layers)

        # Output layer delta
        y_pred = self.a[-1]  # final output, post activation
        deltas[-1] = self.loss_fn.output_delta(
            y_pred, y_true, self.z[-1], self.activations[-1]
        )

        # Hidden layers
        for i in reversed(range(len(self.layers) - 1)):
            W_next = self.layers[i + 1].W
            delta_next = deltas[i + 1]
            z = self.z[i]

            deltas[i] = (delta_next @ W_next.T) * self.activations[i].derivative(z)

        # Compute gradients + update per sample
        for i in range(len(self.layers)):
            layer = self.layers[i]
            delta = deltas[i]

            x = self.a[i]

            dW = x.reshape(-1, 1) @ delta.reshape(1, -1)  # x * delta
            db = delta

            # SGD update after each training sample
            layer.W -= self.lr * dW
            layer.b -= self.lr * db.reshape(-1)

    def train(self, X, Y, epochs=10, print_interval=2):
        for epoch in range(epochs):
            loss_sum = 0
            for x, y in zip(X, Y):
                x = x.reshape(1, -1)
                y = y.reshape(1, -1)

                y_pred = self.forward(x)
                loss_sum += self.loss_fn.loss(y_pred, y)

                self.backward(y)

            if (epoch + 1) % print_interval == 0:
                print(f"Epoch {epoch+1}, Loss: {loss_sum/len(X)}")

    def __str__(self):
        network_architecture = ""
        for layer, activation in zip(self.layers, self.activations):
            network_architecture += f"Linear Block {layer.shape}, Activation: {activation.__name__ if activation else 'None'}\n"

        return (
            f"Network Details:\n"
            f"Input Size: {self.layers[0].shape[0]}\n"
            f"Output Size: {self.layers[-1].shape[-1]}\n"
            f"Architecture:\n{network_architecture}"
            f"Learning Rate: {self.lr}\n"
            f"Weight Initialization Type: {self.weights_type}\n"
        )

### Example 1

In [131]:
# -------------------------
# Example usage (XOR)
# -------------------------


X = np.array([[0,0],
                [0,1],
                [1,0],
                [1,1]])

Y = np.array([[0],
                [1],
                [1],
                [0]])

model = MLP(
    layer_sizes=[2, 4, 1],
    activations=[ReLU, Sigmoid],
    loss="mse",
    lr=0.1
)


In [78]:
print(model)

Network Details:
Input Size: 2
Output Size: 1
Architecture:
Linear Block (2, 4), Activation: ReLU
Linear Block (4, 1), Activation: Sigmoid
Learning Rate: 0.1
Weights Initialization Type: xavier



In [74]:
model.forward([1,0])

array([0.50616343])

In [68]:
model.train(X, Y, epochs=500, print_interval=100)

Epoch 100, Loss: 0.25205775079167114
Epoch 200, Loss: 0.2505113633949398
Epoch 300, Loss: 0.23562676829422807
Epoch 400, Loss: 0.19646206676280437
Epoch 500, Loss: 0.17214178962593962


In [69]:
model.forward([1,0])

array([0.82496124])

### Example 2

In [70]:
import pandas as pd

In [159]:
df = pd.read_csv('seeds_dataset.csv')
df = df.sample(frac=1)

In [160]:
df.head(1)

,var1,var2,var3,var4,var5,var6,var7,class
69,12.73,13.75,0.8458,5.412,2.882,3.533,5.067,1


In [161]:
y = df['class']
X = df.drop('class', axis=1)

In [162]:
y = pd.get_dummies(y).astype('int')

In [163]:
y = y.to_numpy()
X = X.to_numpy()

In [184]:
model = MLP(
    layer_sizes=[7, 16, 32, 16, 8, 3],
    activations=[ReLU, ReLU, ReLU, ReLU, Softmax],
    loss="cross_entropy",
    lr=0.0001
)

In [185]:
# print(model)

In [186]:
model.train(X, y, epochs=50, print_interval=10)

Epoch 10, Loss: 1.0179603951470233
Epoch 20, Loss: 0.9736378981034982
Epoch 30, Loss: 0.9064127023474425
Epoch 40, Loss: 0.8302743923536036
Epoch 50, Loss: 0.7237237090389417


In [190]:
model.forward(X[130])

array([0.42404751, 0.48110287, 0.09484962])

In [189]:
y[130]

array([0, 0, 1])

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 168 to 175
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   var1    210 non-null    float64
 1   var2    210 non-null    float64
 2   var3    210 non-null    float64
 3   var4    210 non-null    float64
 4   var5    210 non-null    float64
 5   var6    210 non-null    float64
 6   var7    210 non-null    float64
 7   class   210 non-null    int64  
dtypes: float64(7), int64(1)
memory usage: 14.8 KB
